In [ ]:
import pandas as pd
import pypsa
import numpy
import pickle
import numpy as np
pd.set_option('display.max_rows', None)

In [ ]:
network = {}
horizon = [2025, 2030, 2035, 2040, 2045, 2050]
path = 'results/baseline/'
for i in horizon:
    network[i] = pypsa.Network(f'{path}networks/base_s_33__2H_{i}.nc')
    network[i].buses.loc[network[i].buses.index.str.contains('EU'),'country'] = 'EU'

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

colors = ['#2196F3', '#E91E63', '#4CAF50', '#FF9800']

import os

if not os.path.exists(f'{path}plots/'):
    os.mkdir(f'{path}plots/')


---
---
### $\text{CO}_2\text{ emissions}$ 
---
---

Positive efficiency: it gives \
Negative efficiency: it consumes 

Positive p_i: taking from bus_i \
Negative p_i: supplying to bus_i

The total emissions from links should be 2.0991e+09

In [ ]:
def get_co2(n):
    
    co2_links_bus1 = n.links[
        n.links.bus1.str.contains('co2 atmosphere', case=False, na=False) 
    ]
    co2_links_bus2 = n.links[
        n.links.bus2.str.contains('co2 atmosphere', case=False, na=False) 
    ]
    co2_links_bus3 = n.links[
        n.links.bus3.str.contains('co2 atmosphere', case=False, na=False) 
    ]

    # CO2 from bus1 links
    if not co2_links_bus1.empty:
        co2_bus1 = (
            -n.links_t.p1[co2_links_bus1.index]
            .multiply(n.snapshot_weightings.generators, axis=0)
        )
    # CO2 from bus2 links
    if not co2_links_bus2.empty:
        co2_bus2 = (
            -n.links_t.p2[co2_links_bus2.index]
            .multiply(n.snapshot_weightings.generators, axis=0)
        )
    # CO2 from bus3 links
    if not co2_links_bus3.empty:
        co2_bus3 = (
            -n.links_t.p3[co2_links_bus3.index]
            .multiply(n.snapshot_weightings.generators, axis=0)
        )

    co2_all_buses = pd.concat((co2_bus1, co2_bus2, co2_bus3), axis=1).sum(axis = 0)

    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    co2_dict = {country: 0 for country in countries_list}

    for i in co2_all_buses.index:
        co2_dict[i[:2]] += co2_all_buses.loc[i]

    return co2_dict

In [ ]:
co2_dict_all_years = {year : {} for year in horizon}
for year in horizon:
    co2_dict_all_years[year] = get_co2(network[year])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: Countries of interest ---
countries_of_interest = ['BE', 'FR', 'GB']
for i, country in enumerate(countries_of_interest):
    co2_country = [co2_dict[country] for co2_dict in co2_dict_all_years.values()]
    ax1.plot(horizon, [v / 1e6 for v in co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
ax1.set_title('CO₂ Emissions by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
co2_EU = [sum(co2_dict.values()) for co2_dict in co2_dict_all_years.values()]
ax2.plot(horizon, [v / 1e6 for v in co2_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
ax2.set_title('EU Total CO₂ Emissions', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'CO₂ Emissions Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{path}plots/{title.replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
plt.show()

---
---
### $\text{Electricity Prices}$ 
---
---

In [ ]:
def get_electricity(n):
    electricity_buses_names = n.buses[(n.buses.carrier == 'AC')].index

    electricity_prices_per_bus = n.buses_t['marginal_price'][electricity_buses_names].mean( axis = 0)
    
    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']

    electricity_prices_per_country_dict = {country: [] for country in countries_list}

    electricity_prices_per_bus_dict = dict(electricity_prices_per_bus)

    for i in electricity_prices_per_bus.index:
        electricity_prices_per_country_dict[i[:2]].append(electricity_prices_per_bus.loc[i])
    for key, item in electricity_prices_per_country_dict.items():
        electricity_prices_per_country_dict[key] = np.mean(item)

    return electricity_prices_per_country_dict, electricity_prices_per_bus_dict

In [ ]:
electricity_per_bus_dict_all_years     = {year : {} for year in horizon}
electricity_per_country_dict_all_years = {year : {} for year in horizon}

for year in horizon:
    electricity_per_country_dict_all_years[year], electricity_per_bus_dict_all_years[year] = get_electricity(network[year])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: Countries of interest ---
countries_of_interest = ['BE', 'FR', 'GB']
for i, country in enumerate(countries_of_interest):
    electricity_country = [electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years.values()]
    ax1.plot(horizon, [v for v in electricity_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Electricity Prices [€/MWh]', fontsize=12)
ax1.set_title('Electricity Prices by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
electricity_EU = [np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years.values()]
ax2.plot(horizon, [v for v in electricity_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Electricity Prices [€/MWh]', fontsize=12)
ax2.set_title('EU Electricity Prices', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Electricity Prices Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{path}plots/{title.replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# def get_electricity_metrics(n):
#     electricity_buses_names = n.buses[(n.buses.carrier == 'AC')].index

#     electricity_prices_per_bus = n.buses_t['marginal_price'][electricity_buses_names]
    
#     countries_list = n.buses.country.unique()
#     countries_list = countries_list[countries_list != '']
#     countries_list = countries_list[countries_list != 'EU']

#     df = pd.DataFrame(columns = list(countries_list) + ['EU'], index = ['electricity_price_zero_hours', "electricity_price_mean", "electricity_price_std"])

#     for country in countries_list:
#         country_buses = [bus_name for bus_name in electricity_buses_names if country in bus_name]
#         prices = electricity_prices_per_bus[country_buses]
#         df.loc['electricity_price_zero_hours', country] = prices.where(prices < 0.1).count().sum() / prices.size
#         df.loc['electricity_price_mean', country] = prices.unstack().mean()
#         df.loc['electricity_price_std', country] = prices.unstack().std()
    
#     df.loc['electricity_price_zero_hours', 'EU'] = electricity_prices_per_bus.where(electricity_prices_per_bus < 0.1).count().sum() / electricity_prices_per_bus.size
#     df.loc['electricity_price_mean', 'EU'] = electricity_prices_per_bus.unstack().mean()
#     df.loc['electricity_price_std', 'EU'] = electricity_prices_per_bus.unstack().std()


#     return df

In [ ]:
#electricity_metrics = {}
#for year in horizon:
#    print(year)
#    electricity_metrics[year] = get_electricity_metrics(network[year])

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# countries_of_interest = ['BE', 'FR', 'GB', 'EU']

# x = np.arange(len(horizon))
# width = 0.8 / len(countries_of_interest)

# fig, ax = plt.subplots(figsize=(10, 6))

# for i, country in enumerate(countries_of_interest):
#     values = []
#     for year in horizon:
#         values.append(electricity_metrics[year].loc['electricity_price_zero_hours', country])

#     ax.bar(x + i * width, values, width=width,
#            label=country, color=colors[i])

# ax.set_xticks(x + width * (len(countries_of_interest) - 1) / 2)
# ax.set_xticklabels(horizon)
# ax.set_xlabel('Year', fontsize=12)
# ax.set_ylabel('Zero price hours', fontsize=12)
# ax.set_title('Electricity Zero Price Hours by Country', fontsize=14, fontweight='bold')
# ax.legend(fontsize=10)
# ax.grid(alpha=0.3, linestyle='--', axis='y')
# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)

# plt.tight_layout()
# plt.show()

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np
# from scipy.stats import norm

# countries_of_interest = ['BE', 'FR', 'GB', 'EU']

# fig, axes = plt.subplots( 1, len(horizon), figsize=(6 * len(horizon), 6))

# for ax, year in zip(axes, horizon):
#     metrics = electricity_metrics[year]

#     x = np.linspace(
#         np.min(metrics.loc['electricity_price_mean', countries_of_interest] - 2 * metrics.loc['electricity_price_std', countries_of_interest]),
#         np.max(metrics.loc['electricity_price_mean',  countries_of_interest] + 2 * metrics.loc['electricity_price_std',  countries_of_interest]),
#         500
#     )

#     for i, country in enumerate(countries_of_interest):
#         mean = metrics.loc['electricity_price_mean', country]
#         std  = metrics.loc['electricity_price_std', country]
#         y = norm.pdf(x, mean, std)
#         label = f"{country} (μ={mean:.1f}, σ={std:.1f})"
#         ax.plot(x, y, linewidth=2.5, color=colors[i], label=label)
#         ax.axvline(mean, color=colors[i], linestyle='--', linewidth=1, alpha=0.5)

#     ax.legend(fontsize=9, framealpha=0.9, loc='upper right')
#     ax.set_title(str(year), fontsize=13, fontweight='bold')
#     ax.set_xlabel('Electricity price [€/MWh]', fontsize=11)
#     ax.grid(alpha=0.3, linestyle='--')
#     ax.spines['top'].set_visible(False)
#     ax.spines['right'].set_visible(False)

# axes[0].set_ylabel('Probability density', fontsize=11)


# plt.suptitle('Electricity Price Distribution by Country and Year',
#              fontsize=15, fontweight='bold', y=1.02)
# plt.tight_layout()
# plt.show()

---
---
### $\text{Molecules Prices}$ 
---
---

In [ ]:
def get_hydrogen(n):
    hydrogen_buses_names = n.buses[(n.buses.carrier == 'H2')].index

    hydrogen_prices_per_bus = n.buses_t['marginal_price'][hydrogen_buses_names].mean( axis = 0)
    
    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']

    hydrogen_prices_per_country_dict = {country: [] for country in countries_list}

    hydrogen_prices_per_bus_dict = dict(hydrogen_prices_per_bus)

    for i in hydrogen_prices_per_bus.index:
        hydrogen_prices_per_country_dict[i[:2]].append(hydrogen_prices_per_bus.loc[i])
    for key, item in hydrogen_prices_per_country_dict.items():
        hydrogen_prices_per_country_dict[key] = np.mean(item)

    return hydrogen_prices_per_country_dict, hydrogen_prices_per_bus_dict

def get_methanol(n):
    methanol_buses_names = n.buses[(n.buses.carrier == 'methanol')].index

    methanol_prices_per_bus = n.buses_t['marginal_price'][methanol_buses_names].mean( axis = 0)
    
    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    #countries_list = countries_list[countries_list != 'EU']

    methanol_prices_per_country_dict = {country: [] for country in countries_list}

    methanol_prices_per_bus_dict = dict(methanol_prices_per_bus)

    for i in methanol_prices_per_bus.index:
        methanol_prices_per_country_dict[i[:2]].append(methanol_prices_per_bus.loc[i])
    for key, item in methanol_prices_per_country_dict.items():
        methanol_prices_per_country_dict[key] = np.mean(item)

    return methanol_prices_per_country_dict, methanol_prices_per_bus_dict

def get_ammonia(n):
    ammonia_buses_names = n.buses[(n.buses.carrier == 'NH3')].index

    ammonia_prices_per_bus = n.buses_t['marginal_price'][ammonia_buses_names].mean( axis = 0)
    
    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    # countries_list = countries_list[countries_list != 'EU']

    ammonia_prices_per_country_dict = {country: [] for country in countries_list}

    ammonia_prices_per_bus_dict = dict(ammonia_prices_per_bus)

    for i in ammonia_prices_per_bus.index:
        ammonia_prices_per_country_dict[i[:2]].append(ammonia_prices_per_bus.loc[i])
    for key, item in ammonia_prices_per_country_dict.items():
        ammonia_prices_per_country_dict[key] = np.mean(item)

    return ammonia_prices_per_country_dict, ammonia_prices_per_bus_dict

def get_oil(n):
    oil_buses_names = n.buses[(n.buses.carrier == 'oil')].index

    oil_prices_per_bus = n.buses_t['marginal_price'][oil_buses_names].mean( axis = 0)
    
    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    #countries_list = countries_list[countries_list != 'EU']

    oil_prices_per_country_dict = {country: [] for country in countries_list}

    oil_prices_per_bus_dict = dict(oil_prices_per_bus)

    for i in oil_prices_per_bus.index:
        oil_prices_per_country_dict[i[:2]].append(oil_prices_per_bus.loc[i])
    for key, item in oil_prices_per_country_dict.items():
        oil_prices_per_country_dict[key] = np.mean(item)

    return oil_prices_per_country_dict, oil_prices_per_bus_dict
	
def get_naphta(n):
    naphta_buses_names = n.buses[(n.buses.carrier == 'naphta')].index

    naphta_prices_per_bus = n.buses_t['marginal_price'][naphta_buses_names].mean( axis = 0)
    
    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']

    naphta_prices_per_country_dict = {country: [] for country in countries_list}

    naphta_prices_per_bus_dict = dict(naphta_prices_per_bus)

    for i in naphta_prices_per_bus.index:
        naphta_prices_per_country_dict[i[:2]].append(naphta_prices_per_bus.loc[i])
    for key, item in naphta_prices_per_country_dict.items():
        naphta_prices_per_country_dict[key] = np.mean(item)

    return naphta_prices_per_country_dict, naphta_prices_per_bus_dict
	
def get_gas(n):
    gas_buses_names = n.buses[(n.buses.carrier == 'gas')].index

    gas_prices_per_bus = n.buses_t['marginal_price'][gas_buses_names].mean( axis = 0)
    
    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']

    gas_prices_per_country_dict = {country: [] for country in countries_list}

    gas_prices_per_bus_dict = dict(gas_prices_per_bus)

    for i in gas_prices_per_bus.index:
        gas_prices_per_country_dict[i[:2]].append(gas_prices_per_bus.loc[i])
    for key, item in gas_prices_per_country_dict.items():
        gas_prices_per_country_dict[key] = np.mean(item)

    return gas_prices_per_country_dict, gas_prices_per_bus_dict


In [ ]:
hydrogen_per_bus_dict_all_years     = {year : {} for year in horizon}
hydrogen_per_country_dict_all_years = {year : {} for year in horizon}

methanol_per_bus_dict_all_years     = {year : {} for year in horizon}
methanol_per_country_dict_all_years = {year : {} for year in horizon}

ammonia_per_bus_dict_all_years     = {year : {} for year in horizon}
ammonia_per_country_dict_all_years = {year : {} for year in horizon}

gas_per_bus_dict_all_years     = {year : {} for year in horizon}
gas_per_country_dict_all_years = {year : {} for year in horizon}

oil_per_bus_dict_all_years     = {year : {} for year in horizon}
oil_per_country_dict_all_years = {year : {} for year in horizon}

naphta_per_bus_dict_all_years     = {year : {} for year in horizon}
naphta_per_country_dict_all_years = {year : {} for year in horizon}

for year in horizon:
    hydrogen_per_country_dict_all_years[year], hydrogen_per_bus_dict_all_years[year] = get_hydrogen(network[year])
    methanol_per_country_dict_all_years[year], methanol_per_bus_dict_all_years[year] = get_methanol(network[year])
    ammonia_per_country_dict_all_years[year],   ammonia_per_bus_dict_all_years[year] = get_ammonia(network[year])
    gas_per_country_dict_all_years[year],           gas_per_bus_dict_all_years[year] = get_gas(network[year])
    oil_per_country_dict_all_years[year],           oil_per_bus_dict_all_years[year] = get_oil(network[year])
    naphta_per_country_dict_all_years[year],     naphta_per_bus_dict_all_years[year] = get_naphta(network[year])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: Countries of interest ---
countries_of_interest = ['BE', 'FR', 'GB']
for i, country in enumerate(countries_of_interest):
    hydrogen_country = [hydrogen_dict[country] for hydrogen_dict in hydrogen_per_country_dict_all_years.values()]
    ax1.plot(horizon, [v for v in hydrogen_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Hydrogen Prices [€/MWh]', fontsize=12)
ax1.set_title('Hydrogen Prices by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
hydrogen_EU = [np.mean(list(hydrogen_dict.values())) for hydrogen_dict in hydrogen_per_bus_dict_all_years.values()]
ax2.plot(horizon, [v for v in hydrogen_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Hydrogen Prices [€/MWh]', fontsize=12)
ax2.set_title('EU Hydrogen Prices', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Hydrogen Prices Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{path}plots/{title.replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: Countries of interest ---
countries_of_interest = ['BE', 'FR', 'GB']
for i, country in enumerate(countries_of_interest):
    methanol_country = [methanol_dict[country] for methanol_dict in methanol_per_country_dict_all_years.values()]
    ax1.plot(horizon, [v for v in methanol_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Methanol Prices [€/MWh]', fontsize=12)
ax1.set_title('Methanol Prices by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
methanol_EU = [np.mean(list(methanol_dict.values())) for methanol_dict in methanol_per_bus_dict_all_years.values()]
ax2.plot(horizon, [v for v in methanol_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Methanol Prices [€/MWh]', fontsize=12)
ax2.set_title('EU Methanol Prices', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Methanol Prices Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{path}plots/{title.replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: Countries of interest ---
countries_of_interest = ['BE', 'FR', 'GB']
for i, country in enumerate(countries_of_interest):
    ammonia_country = [ammonia_dict[country] for ammonia_dict in ammonia_per_country_dict_all_years.values()]
    ax1.plot(horizon, [v for v in ammonia_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Ammonia Prices [€/MWh]', fontsize=12)
ax1.set_title('Ammonia Prices by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
ammonia_EU = [np.mean(list(ammonia_dict.values())) for ammonia_dict in ammonia_per_bus_dict_all_years.values()]
ax2.plot(horizon, [v for v in ammonia_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Ammonia Prices [€/MWh]', fontsize=12)
ax2.set_title('EU Ammonia Prices', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Ammonia Prices Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{path}plots/{title.replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: Countries of interest ---
countries_of_interest = ['BE', 'FR', 'GB']
for i, country in enumerate(countries_of_interest):
    oil_country = [oil_dict[country] for oil_dict in oil_per_country_dict_all_years.values()]
    ax1.plot(horizon, [v for v in oil_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Oil Prices [€/MWh]', fontsize=12)
ax1.set_title('Oil Prices by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
oil_EU = [np.mean(list(oil_dict.values())) for oil_dict in oil_per_bus_dict_all_years.values()]
ax2.plot(horizon, [v for v in oil_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Oil Prices [€/MWh]', fontsize=12)
ax2.set_title('EU Oil Prices', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Oil Prices Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{path}plots/{title.replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: Countries of interest ---
countries_of_interest = ['BE', 'FR', 'GB']
for i, country in enumerate(countries_of_interest):
    gas_country = [gas_dict[country] for gas_dict in gas_per_country_dict_all_years.values()]
    ax1.plot(horizon, [v for v in gas_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Gas Prices [€/MWh]', fontsize=12)
ax1.set_title('Gas Prices by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
gas_EU = [np.mean(list(gas_dict.values())) for gas_dict in gas_per_bus_dict_all_years.values()]
ax2.plot(horizon, [v for v in gas_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Gas Prices [€/MWh]', fontsize=12)
ax2.set_title('EU Gas Prices', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Gas Prices Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{path}plots/{title.replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: Countries of interest ---
countries_of_interest = ['BE', 'FR', 'GB']
for i, country in enumerate(countries_of_interest):
    naphta_country = [naphta_dict[country] for naphta_dict in naphta_per_country_dict_all_years.values()]
    ax1.plot(horizon, [v for v in naphta_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Naphta Prices [€/MWh]', fontsize=12)
ax1.set_title('Naphta Prices by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
naphta_EU = [np.mean(list(naphta_dict.values())) for naphta_dict in naphta_per_bus_dict_all_years.values()]
ax2.plot(horizon, [v for v in naphta_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Naphta Prices [€/MWh]', fontsize=12)
ax2.set_title('EU Naphta Prices', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Naphta Prices Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{path}plots/{title.replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# def get_hydrogen_metrics(n):
#     hydrogen_buses_names = n.buses[(n.buses.carrier == 'H2')].index

#     hydrogen_prices_per_bus = n.buses_t['marginal_price'][hydrogen_buses_names]
    
#     countries_list = n.buses.country.unique()
#     countries_list = countries_list[countries_list != '']
#     countries_list = countries_list[countries_list != 'EU']

#     df = pd.DataFrame(columns = list(countries_list) + ['EU'], index = ['hydrogen_price_zero_hours', "hydrogen_price_mean", "hydrogen_price_std"])

#     for country in countries_list:
#         country_buses = [bus_name for bus_name in hydrogen_buses_names if country in bus_name]
#         prices = hydrogen_prices_per_bus[country_buses]
#         df.loc['hydrogen_price_zero_hours', country] = prices.where(prices < 0.1).count().sum() / prices.size
#         df.loc['hydrogen_price_mean', country] = prices.unstack().mean()
#         df.loc['hydrogen_price_std', country] = prices.unstack().std()
    
#     df.loc['hydrogen_price_zero_hours', 'EU'] = hydrogen_prices_per_bus.where(hydrogen_prices_per_bus < 0.1).count().sum() / hydrogen_prices_per_bus.size
#     df.loc['hydrogen_price_mean', 'EU'] = hydrogen_prices_per_bus.unstack().mean()
#     df.loc['hydrogen_price_std', 'EU'] = hydrogen_prices_per_bus.unstack().std()


#     return df

In [ ]:
# hydrogen_metrics = {}
# for year in horizon:
#     print(year)
#     hydrogen_metrics[year] = get_hydrogen_metrics(network[year])

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np
# from scipy.stats import norm

# countries_of_interest = ['BE', 'FR', 'GB', 'EU']

# fig, axes = plt.subplots( 1, len(horizon), figsize=(6 * len(horizon), 6))

# for ax, year in zip(axes, horizon):
#     metrics = hydrogen_metrics[year]

#     x = np.linspace(
#         np.min(metrics.loc['hydrogen_price_mean', countries_of_interest] - 2 * metrics.loc['hydrogen_price_std', countries_of_interest]),
#         np.max(metrics.loc['hydrogen_price_mean',  countries_of_interest] + 2 * metrics.loc['hydrogen_price_std',  countries_of_interest]),
#         500
#     )

#     for i, country in enumerate(countries_of_interest):
#         mean = metrics.loc['hydrogen_price_mean', country]
#         std  = metrics.loc['hydrogen_price_std', country]
#         y = norm.pdf(x, mean, std)
#         label = f"{country} (μ={mean:.1f}, σ={std:.1f})"
#         ax.plot(x, y, linewidth=2.5, color=colors[i], label=label)
#         ax.axvline(mean, color=colors[i], linestyle='--', linewidth=1, alpha=0.5)

#     ax.legend(fontsize=9, framealpha=0.9, loc='upper right')
#     ax.set_title(str(year), fontsize=13, fontweight='bold')
#     ax.set_xlabel('Hydrogen price [€/MWh]', fontsize=11)
#     ax.grid(alpha=0.3, linestyle='--')
#     ax.spines['top'].set_visible(False)
#     ax.spines['right'].set_visible(False)

# axes[0].set_ylabel('Probability density', fontsize=11)


# plt.suptitle('Hydrogen Price Distribution by Country and Year',
#              fontsize=15, fontweight='bold', y=1.02)
# plt.tight_layout()
# plt.show()

---
---
### $\text{LCOH and Green Hydrogen metrics}$ 
---
---

In [ ]:
def get_lcoh_grid_connected(n):

    weights = n.snapshot_weightings.generators

    # 1. Isolate the electrolyzer links
    electrolyzers = n.links[n.links.carrier == 'H2 Electrolysis']
    elec_indices = electrolyzers.index

    # 2. Calculate Annualized Fixed Costs (CAPEX + Fixed O&M) per link
    # capital_cost in PyPSA is already annualized per MW
    fixed_costs = electrolyzers['capital_cost'] * electrolyzers['p_nom_opt']

    # 3. Calculate Variable input costs (Electricity consumed * Local Nodal Price)
    # Extract hourly electricity prices for only those specific buses
    link_to_bus_map = n.links.loc[elec_indices, 'bus0']
    aligned_prices = n.buses_t.marginal_price[link_to_bus_map]
    aligned_prices.columns = elec_indices
    # Extract hourly power consumption at bus0 for each electrolyzer
    hourly_consumption = n.links_t.p0[elec_indices]


    # Multiply element-wise (hourly) and sum over the year to get total variable cost per link
    variable_costs = (hourly_consumption.mul(weights, axis=0) * aligned_prices).sum(axis=0)

    # 4. Calculate total annual hydrogen production per link (Output at bus1)
    # Note: If your model tracks efficiency losses, p1 is the actual H2 generated
    annual_h2_produced = -n.links_t.p1[elec_indices].mul(weights, axis=0).sum(axis=0)

    # 5. Calculate LCOH per individual link
    # Avoid division by zero for links that weren't built (p_nom_opt == 0)
    lcoh_per_link = (fixed_costs + variable_costs) / annual_h2_produced
    lcoh_per_link = lcoh_per_link.dropna()  # Drops links that were not optimized into existence

    # Convert to a clean DataFrame for analysis or plotting
    df_lcoh = pd.DataFrame({
        'Bus_Location': electrolyzers.loc[lcoh_per_link.index, 'bus1'],
        'Capacity': electrolyzers.loc[lcoh_per_link.index, 'p_nom_opt'],
        'H2 Produced' : annual_h2_produced,
        'LCOH': lcoh_per_link
    })

    return df_lcoh

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

countries_of_interest = ['BE', 'FR', 'GB']
colors = ['#2196F3','#E91E63','#4CAF50','#FF9800']

# Collect data across years
h2_produced_all  = {c: [] for c in countries_of_interest}
h2_produced_all['EU'] = []
lcoh_all         = {c: [] for c in countries_of_interest}
lcoh_all['EU'] = []

for year in horizon:
    df_lcoh = get_lcoh_grid_connected(network[year])

    h2_produced = {country: 0 for country in countries_of_interest}
    lcoh        = {country: [] for country in countries_of_interest}

    for i in df_lcoh.index:
        if i[:2] in h2_produced:
            h2_produced[i[:2]] += df_lcoh.loc[i, 'H2 Produced']
            lcoh[i[:2]].append(df_lcoh.loc[i, 'LCOH'])

    for key in lcoh:
        lcoh[key] = np.mean(lcoh[key])

    h2_produced['EU'] = df_lcoh['H2 Produced'].sum()
    lcoh['EU']        = np.mean(df_lcoh['LCOH'])

    for c in countries_of_interest + ['EU']:
        h2_produced_all[c].append(h2_produced[c])
        lcoh_all[c].append(lcoh[c])

x = np.arange(len(horizon))
width = 0.8 / len(countries_of_interest)

# ── Figure 1: H2 Production ──────────────────────────────────────────
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, h2_produced_all[country],
            color=colors[i], marker='o', linewidth=2.5, label=country)
ax1.set_title('H₂ Production by Country', fontweight='bold')
ax1.set_ylabel('H₂ Produced [MWh]')
ax1.set_xlabel('Year')
ax1.legend()
ax1.grid(alpha=0.3, linestyle='--', axis='y')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2.plot(horizon, h2_produced_all['EU'], color=colors[-1],
         marker='o', linewidth=2.5, label='EU')
ax2.set_title('EU Total H₂ Production', fontweight='bold')
ax2.set_ylabel('H₂ Produced [MWh]')
ax2.set_xlabel('Year')
ax2.legend()
ax2.grid(alpha=0.3, linestyle='--', axis='y')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('Hydrogen Production', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Figure 2: LCOH ───────────────────────────────────────────────────
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 5))

for i, country in enumerate(countries_of_interest):
    ax3.plot(horizon, lcoh_all[country],
            color=colors[i], marker='o', linewidth=2.5, label=country)
ax3.set_title('LCOH by Country', fontweight='bold')
ax3.set_ylabel('LCOH [€/MWh]')
ax3.set_xlabel('Year')
ax3.legend()
ax3.grid(alpha=0.3, linestyle='--', axis='y')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

ax4.plot(horizon, lcoh_all['EU'], color=colors[-1],
         marker='o', linewidth=2.5, label='EU')
ax4.set_title('EU Average LCOH', fontweight='bold')
ax4.set_ylabel('LCOH [€/MWh]')
ax4.set_xlabel('Year')
ax4.legend()
ax4.grid(alpha=0.3, linestyle='--', axis='y')
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)

plt.suptitle('Levelized Cost of Hydrogen (LCOH)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
---
### $\text{Share of Renewable Electricity}$ 
---
---

<i>Definition Renewable electricity: </i> renewable sources, as defined in Article 2, point (1) of Directive (EU) 2018/2001, excluding units producing electricity from biomass and storage units

<i> renewable sources, as defined in Article 2, point (1) of Directive (EU) 2018/2001 :</i> energy from renewable non-fossil sources, namely wind, solar (solar thermal and solar photovoltaic) and geothermal energy, ambient energy, tide, wave and other ocean energy, hydropower, biomass, landfill gas, sewage treatment plant gas, and biogas.